In [1]:
import os
import re
import numpy as np
import pandas as pd
import Levenshtein
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Create required output directories
os.makedirs("output", exist_ok=True)
os.makedirs("output/cleaned_data", exist_ok=True)

In [2]:
# Load Pre-Cleaned Training Data
train_s1 = pd.read_csv("output/cleaned_data/train_source1_cleaned.tsv", sep="\t")
train_s2 = pd.read_csv("output/cleaned_data/train_source2_cleaned.tsv", sep="\t")
train_s3 = pd.read_csv("output/cleaned_data/train_source3_cleaned.tsv", sep="\t")
train_gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")

# Load Pre-Cleaned Test Data
test_s1 = pd.read_csv("output/cleaned_data/test_source1_cleaned.tsv", sep="\t")
test_s2 = pd.read_csv("output/cleaned_data/test_source2_cleaned.tsv", sep="\t")
test_s3 = pd.read_csv("output/cleaned_data/test_source3_cleaned.tsv", sep="\t")

# Ensure all text columns are treated as standard strings for similarity functions
for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    df['clean_name'] = df['clean_name'].astype(str)
    df['clean_address'] = df['clean_address'].astype(str)

print("Datasets loaded successfully and ready for candidate generation!")

Datasets loaded successfully and ready for candidate generation!


In [6]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def generate_candidates_fast_check(s1_df, s2_df, s3_df, threshold=0.20, s1_chunk_size=500, pool_chunk_size=50000, max_s1_records=1000, max_pool_records=10000):
    candidate_pairs = []
    
    # Restrict both S1 and Pool for an instant sanity check
    if max_s1_records is not None:
        print(f"--- [FAST SANITY CHECK ACTIVE] Restricting S1 to {max_s1_records} and Pool to {max_pool_records} records ---")
        s1_df = s1_df.head(max_s1_records)
        
    countries = s1_df['country'].dropna().unique()
    
    for country in countries:
        s1_sub = s1_df[s1_df['country'] == country].copy()
        s2_sub = s2_df[s2_df['country'] == country].copy()
        s3_sub = s3_df[s3_df['country'] == country].copy()
        
        if s1_sub.empty:
            continue
        pool_df = pd.concat([s2_sub, s3_sub], ignore_index=True)
        if pool_df.empty:
            continue
            
        # Restrict pool size if sanity check mode is active
        if max_pool_records is not None:
            pool_df = pool_df.head(max_pool_records)
            
        s1_sub['clean_name'] = s1_sub['clean_name'].fillna("").astype(str)
        pool_df['clean_name'] = pool_df['clean_name'].fillna("").astype(str)
        
        print(f"Processing country '{country}': {len(s1_sub)} S1 records vs {len(pool_df)} pool records...")
        
        # Sample strings safely for vectorizer
        s1_sample = s1_sub['clean_name'].sample(n=min(5000, len(s1_sub)), random_state=42).tolist()
        pool_sample = pool_df['clean_name'].sample(n=min(5000, len(pool_df)), random_state=42).tolist()
        
        vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4), max_features=30000)
        vectorizer.fit(s1_sample + pool_sample)
        
        for s1_start in range(0, len(s1_sub), s1_chunk_size):
            s1_chunk = s1_sub.iloc[s1_start:s1_start + s1_chunk_size]
            tfidf_s1_chunk = vectorizer.transform(s1_chunk['clean_name'])
            
            for pool_start in range(0, len(pool_df), pool_chunk_size):
                pool_chunk = pool_df.iloc[pool_start:pool_start + pool_chunk_size]
                tfidf_pool_chunk = vectorizer.transform(pool_chunk['clean_name'])
                
                sim_matrix = cosine_similarity(tfidf_s1_chunk, tfidf_pool_chunk)
                rows, cols = np.where(sim_matrix >= threshold)
                
                for r, c in zip(rows, cols):
                    candidate_pairs.append({
                        'source1_entity_id': s1_chunk.iloc[r]['entity_id'],
                        'candidate_entity_id': pool_chunk.iloc[c]['entity_id']
                    })
                        
    return pd.DataFrame(candidate_pairs)

# Run this - it will finish in about 5 to 10 seconds!
train_candidates = generate_candidates_fast_check(train_s1, train_s2, train_s3, threshold=0.20, max_s1_records=1000, max_pool_records=10000)
print(f"Sanity check passed instantly! Generated {len(train_candidates)} candidate pairs.")

--- [FAST SANITY CHECK ACTIVE] Restricting S1 to 1000 and Pool to 10000 records ---
Processing country 'US': 595 S1 records vs 10000 pool records...
Processing country 'India': 405 S1 records vs 10000 pool records...
Sanity check passed instantly! Generated 58566 candidate pairs.


In [ ]:
def compute_features_chunked(pairs_df, s1_df, pool_df, chunk_size=100000):
    print("Building lookup dictionaries for full dataset...")
    s1_lookup = s1_df.set_index('entity_id')[['clean_name', 'clean_address']].to_dict('index')
    pool_lookup = pool_df.set_index('entity_id')[['clean_name', 'clean_address']].to_dict('index')
    
    total_pairs = len(pairs_df)
    print(f"Computing features for {total_pairs} total pairs in chunks...")
    
    all_chunks = []
    
    for start_idx in range(0, total_pairs, chunk_size):
        chunk = pairs_df.iloc[start_idx:start_idx + chunk_size]
        chunk_features = []
        
        s1_ids = chunk['source1_entity_id'].values
        pool_ids = chunk['candidate_entity_id'].values
        
        for s1_id, pool_id in zip(s1_ids, pool_ids):
            rec1 = s1_lookup.get(s1_id, {'clean_name': '', 'clean_address': ''})
            rec2 = pool_lookup.get(pool_id, {'clean_name': '', 'clean_address': ''})
            
            name1, addr1 = rec1['clean_name'], rec1['clean_address']
            name2, addr2 = rec2['clean_name'], rec2['clean_address']
            
            name_lev = Levenshtein.ratio(name1, name2)
            addr_lev = Levenshtein.ratio(addr1, addr2)
            
            t1 = set(name1.split())
            t2 = set(name2.split())
            name_jaccard = len(t1 & t2) / max(1, len(t1 | t2))
            
            chunk_features.append([name_lev, addr_lev, name_jaccard])
            
        all_chunks.append(np.array(chunk_features))
        print(f"-> Completed batch: {min(start_idx + chunk_size, total_pairs)} / {total_pairs} pairs processed.")
        
    return np.vstack(all_chunks)

# Use this when running on the full dataset
train_pool = pd.concat([train_s2, train_s3], ignore_index=True)
X_train = compute_features_chunked(train_candidates, train_s1, train_pool)
print("Full feature engineering completed safely without memory errors!")

In [ ]:
def compute_features_fast(pairs_df, s1_df, pool_df):
    print("Building fast lookup dictionaries...")
    s1_lookup = s1_df.set_index('entity_id')[['clean_name', 'clean_address']].to_dict('index')
    pool_lookup = pool_df.set_index('entity_id')[['clean_name', 'clean_address']].to_dict('index')
    
    print(f"Computing features for {len(pairs_df)} candidate pairs vector-style...")
    
    features = []
    # Extract values directly as arrays/lists for speed
    s1_ids = pairs_df['source1_entity_id'].values
    pool_ids = pairs_df['candidate_entity_id'].values
    
    for s1_id, pool_id in zip(s1_ids, pool_ids):
        rec1 = s1_lookup.get(s1_id, {'clean_name': '', 'clean_address': ''})
        rec2 = pool_lookup.get(pool_id, {'clean_name': '', 'clean_address': ''})
        
        name1, addr1 = rec1['clean_name'], rec1['clean_address']
        name2, addr2 = rec2['clean_name'], rec2['clean_address']
        
        name_lev = Levenshtein.ratio(name1, name2)
        addr_lev = Levenshtein.ratio(addr1, addr2)
        
        t1 = set(name1.split())
        t2 = set(name2.split())
        name_jaccard = len(t1 & t2) / max(1, len(t1 | t2))
        
        features.append([name_lev, addr_lev, name_jaccard])
        
    return np.array(features)

train_pool = pd.concat([train_s2, train_s3], ignore_index=True)
X_train = compute_features_fast(train_candidates, train_s1, train_pool)
print("Feature engineering completed instantly!")

In [ ]:
gt_dict = {}
for _, row in train_gt.iterrows():
    matches = str(row['matched_entity_ids']).split(',') if pd.notna(row['matched_entity_ids']) else []
    gt_dict[row['source1_entity_id']] = set([m.strip() for m in matches if m.strip()])

y_train = np.array([
    1 if row['candidate_entity_id'] in gt_dict.get(row['source1_entity_id'], set()) else 0
    for _, row in train_candidates.iterrows()
])

clf = lgb.LGBMClassifier(n_estimators=120, learning_rate=0.05, random_state=42)
clf.fit(X_train, y_train)
print("LightGBM model trained successfully!")

In [ ]:
# For actual testing/submission, set max_s1_records=None to process all test records
test_candidates = generate_candidates_safe_with_check(test_s1, test_s2, test_s3, threshold=0.20, max_s1_records=2000)
test_pool = pd.concat([test_s2, test_s3], ignore_index=True)

if not test_candidates.empty:
    X_test = compute_features(test_candidates, test_s1, test_pool)
    probs = clf.predict_proba(X_test)[:, 1]
    test_candidates['is_match'] = (probs >= 0.75).astype(int)
    matched_pairs = test_candidates[test_candidates['is_match'] == 1]
else:
    matched_pairs = pd.DataFrame(columns=['source1_entity_id', 'candidate_entity_id'])

# Save candidate_pairs.tsv
if not test_candidates.empty:
    candidate_agg = test_candidates.groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: ",".join(sorted(list(set(x))))).reset_index()
    candidate_agg.columns = ['source1_entity_id', 'candidate_entity_ids']
else:
    candidate_agg = pd.DataFrame(columns=['source1_entity_id', 'candidate_entity_ids'])

candidate_output = test_s1[['entity_id']].merge(candidate_agg, left_on='entity_id', right_on='source1_entity_id', how='left')
candidate_output['candidate_entity_ids'] = candidate_output['candidate_entity_ids'].fillna("")
candidate_output = candidate_output[['entity_id', 'candidate_entity_ids']].rename(columns={'entity_id': 'source1_entity_id'})
candidate_output.to_csv("output/candidate_pairs.tsv", sep="\t", index=False)

# Save matching_results.tsv
if not matched_pairs.empty:
    match_agg = matched_pairs.groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: ",".join(sorted(list(set(x))))).reset_index()
    match_agg.columns = ['source1_entity_id', 'matched_entity_ids']
else:
    match_agg = pd.DataFrame(columns=['source1_entity_id', 'matched_entity_ids'])

final_output = test_s1[['entity_id']].merge(match_agg, left_on='entity_id', right_on='source1_entity_id', how='left')
final_output['matched_entity_ids'] = final_output['matched_entity_ids'].fillna("")
final_output = final_output[['entity_id', 'matched_entity_ids']].rename(columns={'entity_id': 'source1_entity_id'})
final_output.to_csv("output/matching_results.tsv", sep="\t", index=False)

print("Pipeline execution complete! Output files generated successfully in 'output/'.")

In [ ]:
python3 utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/test